# orquestador_maestro.ipynb

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Módulo** | Orquestador Maestro — todas las fases |

---

## 🎯 Qué hace

Lanza en secuencia los orquestadores de cada fase del TFM.
Cada orquestador de fase ejecuta todos sus notebooks y regenera sus HTMLs.
Al terminar, todos los HTMLs del proyecto están actualizados.

## 📋 Requisitos

- Todos los orquestadores de fase en sus carpetas correspondientes
- `src/utils/orquestador.py` con `ejecutar_notebook()`
- Entorno: `tfm_abandono` (nbconvert, ipywidgets)

## 📤 Genera

Regenera todos los HTMLs de todas las fases.

## ⚠️ Advertencia

Tarda varias horas. Lanzar por la noche o cuando no necesites el ordenador.
Fase 5 (modelado) es la más lenta — carga desde .pkl pero aún así tarda.

## ➡️ Siguiente

Revisar HTMLs en `docs/html/` de cada fase.


In [ ]:
# 1. CONFIGURACIÓN DE RUTAS
import sys
from pathlib import Path

def _encontrar_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError('No se encontró src/ subiendo desde ' + str(start))

ROOT = _encontrar_root(Path.cwd())
sys.path.insert(0, str(ROOT))

DIR_NOTEBOOKS = ROOT / 'notebooks'

print(f'ROOT: {ROOT}')
print(f'DIR_NOTEBOOKS: {DIR_NOTEBOOKS}')


In [ ]:
# 2. IMPORTS
from src.utils.orquestador import ejecutar_notebook


In [ ]:
# 3. LISTA DE ORQUESTADORES POR FASE
# Cada tupla: (nombre_fase, carpeta, fichero_orquestador)
# Editar para incluir/excluir fases o cambiar el orden.

ORQUESTADORES = [
    ('Fase 1 — Transformación',    'fase1_transformacion', 'f1_m00_ejecucion.ipynb'),
    ('Fase 2 — EDA Raw',           'fase2_eda',            'f2_m00_ejecucion.ipynb'),
    ('Fase 3 — Features',          'fase3_features',       'f3_m00_ejecucion.ipynb'),
    ('Pre-Modelado AutoML',        'fase_automl',          'fautoml_m00_ejecucion.ipynb'),
    ('Fase 4 — EDA Final',         'fase4_eda',            'f4_m00_ejecucion.ipynb'),
    ('Fase 5 — Modelado',          'fase5_modelado',       'f5_m00_ejecucion.ipynb'),
    ('Fase 6 — Evaluación',        'fase6_evaluacion',     'f6_m00_ejecucion.ipynb'),
]

TIMEOUT = 3600  # 1 hora por celda

print(f'{len(ORQUESTADORES)} fases configuradas.')
for nombre, carpeta, nb in ORQUESTADORES:
    ruta = DIR_NOTEBOOKS / carpeta / nb
    estado = '✅' if ruta.exists() else '❌ NO ENCONTRADO'
    print(f'  {estado}  {nombre}')


In [ ]:
# 4. EJECUTAR FASES EN SECUENCIA
import time

resultados = []

for nombre_fase, carpeta, fichero in ORQUESTADORES:
    ruta_nb = DIR_NOTEBOOKS / carpeta / fichero
    
    if not ruta_nb.exists():
        print(f'⚠️  {nombre_fase}: orquestador no encontrado — saltando.')
        resultados.append((nombre_fase, False, 'no encontrado'))
        continue
    
    print(f'\n{"="*60}')
    print(f'▶️  Iniciando: {nombre_fase}')
    print(f'{"="*60}')
    
    t_inicio = time.time()
    ok = ejecutar_notebook(ruta_nb, timeout=TIMEOUT)
    t_fin = time.time()
    minutos = (t_fin - t_inicio) / 60
    
    estado = '✅' if ok else '❌'
    print(f'{estado} {nombre_fase} — {minutos:.1f} min')
    resultados.append((nombre_fase, ok, f'{minutos:.1f} min'))

# Resumen final
print(f'\n{"="*60}')
print('RESUMEN FINAL')
print(f'{"="*60}')
n_ok = sum(1 for _, ok, _ in resultados if ok)
for nombre, ok, info in resultados:
    icono = '✅' if ok else '❌'
    print(f'  {icono}  {nombre} — {info}')
print(f'\nCompletadas: {n_ok}/{len(ORQUESTADORES)} fases')
